In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))
for _pkg in ('plotly', 'anywidget'):        # anywidget backs Plotly's FigureWidget
    try:
        __import__(_pkg)
    except ImportError:
        import subprocess; subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg])
import sleepcells as sc, sleepwidgets as sw

INSTANCE      = 'results/sukhumvit_90.json'
QPU_RESULTS   = 'results/headline_qpu_90_results.json'
BASEMAP_PNG   = 'data/sukhumvit_basemap.png'
BASEMAP_JSON  = 'data/sukhumvit_basemap.json'

inst = sc.load_instance(INSTANCE)
res  = sc.load_results(QPU_RESULTS)
sel, ret, tot, defect = sc.decode_shots(res['shots'], inst['n_atoms'], policy='postselect')
asleep, viol, nvalid = sc.best_valid_set(sel, inst['graph'])   # reject blockade violations
print(f"{res['source']}  |  {tot} shots  |  post-selected {ret}  |  {nvalid} valid sleep sets")
print(f"best sleep set: {len(asleep)} of {inst['n_atoms']} cells asleep")

## Is the result a genuine MIS?
Before we display it, prove the returned graph state is a real independent set — the coverage certificate.

In [ ]:
G = inst['graph']; S = set(asleep)
bad = [(u, v) for u, v in G.edges() if u in S and v in S]
addable = [v for v in G if v not in S and not any(nb in S for nb in G.neighbors(v))]
print(f"sleep set size      : {len(S)}   (exact optimum {inst['classical']['exact_size']})")
print(f"independent set     : {'YES - 0 overlapping pairs' if not bad else f'NO ({len(bad)} violations)'}")
print(f"maximal             : {'YES - no cell can be added' if not addable else f'NO - could add {addable}'}")
print(f"coverage certificate: {'COVERAGE VERIFIED' if sc.verify_independent_set(G, S) else 'FAILED'}")

## ①  From district to atoms
Step through the three states with the toggle.

In [ ]:
sw.pipeline_view(inst, BASEMAP_PNG, BASEMAP_JSON)

## ②  Frequency reuse — one MIS per channel

Every channel was **measured on QuEra Aquila** (iterated MIS on the residual sub-register). Step through the channels — each MIS is one frequency; the conflict edges thin out until the whole district runs on a handful of channels.

In [ ]:
import json
assign = json.load(open('results/channel_assignment_90_qpu.json'))   # real QPU coloring
edges  = list(inst['graph'].edges())
print(f"{assign['source']}: {assign['n_channels']} channels "
      f"(optimal {assign['chromatic_number']}, greedy overspend {assign['overspend']}); "
      f"round sizes {assign['round_sizes']}")
sw.channel_stepper_view(assign, edges, BASEMAP_PNG, BASEMAP_JSON)